# 05 — Final held-out GATv2 test evaluation

This notebook evaluates the trained residual GATv2 model on the held-out miniImageNet test split.

It mirrors the existing final-evaluation protocol:

- 600 fixed test episodes
- 5-way / 5-shot from the GATv2 JSON
- 15 queries per class
- test seed `20_000`
- frozen CLS and mean-patch baselines evaluated on the exact same episodes
- the GATv2 checkpoint loaded from the JSON experiment name

Do not use this notebook for architecture or hyperparameter selection.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source:", SRC_DIR)


In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)


## Load the GATv2 configuration

The model and graph are reconstructed from exactly the same JSON used during training.


In [ ]:
CONFIG_PATH = Path("configs/residual_gatv2_5shot.json")
config = json.loads(CONFIG_PATH.read_text())

required_keys = [
    "experiment_name",
    "n_way",
    "k_shot",
    "input_dim",
    "hidden_dim",
    "num_layers",
    "attention_heads",
    "edge_dim",
    "dropout",
    "top_k",
    "graph_temperature",
    "cls_temperature",
    "initial_residual_scale",
    "graph_microbatch_size",
    "eval_queries_per_class",
]

missing = [key for key in required_keys if key not in config]
if missing:
    raise KeyError(f"Missing required GATv2 config keys: {missing}")

TEST_NUM_EPISODES = 600
TEST_SEED = 20_000
MAX_CACHED_SHARDS = 6

print(json.dumps(config, indent=2))


## Restore test features and create the fixed test episodes

These are the same test-episode settings used by the existing final evaluation notebook.


In [ ]:
from cross_image_glot.storage import (
    restore_feature_splits,
    atomic_json_save,
)
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.baselines import evaluate_frozen_baseline
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder

restore_feature_splits(
    ["test"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

test_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "test",
    max_cached_shards=MAX_CACHED_SHARDS,
)

test_episodes = FewShotFeatureEpisodeDataset(
    test_features,
    n_way=config["n_way"],
    k_shot=config["k_shot"],
    queries_per_class=config["eval_queries_per_class"],
    num_episodes=TEST_NUM_EPISODES,
    seed=TEST_SEED,
    vary_by_epoch=False,
)

print("Test episodes:", len(test_episodes))
print(
    f"Protocol: {config['n_way']}-way "
    f"{config['k_shot']}-shot, "
    f"{config['eval_queries_per_class']} queries/class"
)


## Frozen baselines

Both baselines are evaluated on the exact same 600 test episodes used by GATv2.


In [ ]:
cls_metrics = evaluate_frozen_baseline(
    test_episodes,
    "cls",
    device,
    TEST_NUM_EPISODES,
    temperature=config["cls_temperature"],
)

mean_metrics = evaluate_frozen_baseline(
    test_episodes,
    "mean_patch",
    device,
    TEST_NUM_EPISODES,
    temperature=config["cls_temperature"],
)

print("CLS baseline:", cls_metrics)
print("Mean-patch baseline:", mean_metrics)


## Reconstruct GATv2 and load the best checkpoint

Evaluation must instantiate the same encoder, readout, graph construction, and residual wrapper used during training.


In [ ]:
from cross_image_glot.models import (
    PatchGATv2Encoder,
    MeanPrototypeCosineReadout,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)
from cross_image_glot.training import evaluate_residual_dataset

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(test_features.metadata["grid_size"]),
    top_k=config["top_k"],
    min_similarity=None,
    graph_dtype=torch.float32,
    similarity_device=device,
)

encoder = PatchGATv2Encoder(
    input_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    heads=config["attention_heads"],
    edge_dim=config["edge_dim"],
    dropout=config["dropout"],
)

readout = MeanPrototypeCosineReadout(
    temperature=config["graph_temperature"],
    learnable_temperature=False,
)

graph_matcher = CrossImageGraphMatcher(
    encoder=encoder,
    readout=readout,
)

model = BaselinePreservingResidualMatcher(
    graph_matcher,
    config["initial_residual_scale"],
)

checkpoint_path = (
    paths.drive_checkpoint_dir
    / config["experiment_name"]
    / "best.pt"
)

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Best GATv2 checkpoint does not exist: {checkpoint_path}"
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Checkpoint:", checkpoint_path)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print(
    "Best validation accuracy:",
    checkpoint.get("best_validation_accuracy", "unknown"),
)
print(
    "Learned residual scale:",
    float(model.residual_scale.detach().cpu()),
)


## Final test evaluation

The residual GATv2 model is evaluated on all 600 fixed test episodes and compared against the frozen baselines.


In [ ]:
model_metrics = evaluate_residual_dataset(
    model,
    graph_builder,
    test_episodes,
    device,
    TEST_NUM_EPISODES,
    config["graph_microbatch_size"],
    config["cls_temperature"],
    log_interval=20,
    split_name="test",
)

results = {
    "model_kind": "residual_gatv2",
    "experiment_name": config["experiment_name"],
    "protocol": {
        "n_way": config["n_way"],
        "k_shot": config["k_shot"],
        "queries_per_class": config["eval_queries_per_class"],
        "num_episodes": TEST_NUM_EPISODES,
        "seed": TEST_SEED,
    },
    "model": model_metrics.to_dict(),
    "residual_scale": float(model.residual_scale.detach().cpu()),
    "cls_baseline": cls_metrics.to_dict(),
    "mean_patch_baseline": mean_metrics.to_dict(),
}

output = (
    paths.drive_results_dir
    / config["experiment_name"]
    / "test_metrics.json"
)

atomic_json_save(results, output)

print("Residual GATv2:", model_metrics)
print("CLS baseline:", cls_metrics)
print("Mean-patch baseline:", mean_metrics)
print("Residual scale:", model.residual_scale.item())
print("Saved:", output)
